# 01 — Dataset indirme (Faz 1)

**Çalışma zamanı: CPU yeterli.** Runtime → Change runtime type → CPU. GPU kotanı eğitime sakla.

**Depolama planı**
- Çalışma kopyası `/content/raw` altında: hızlı ama oturum kapanınca silinir. Eğitim ve audit her zaman buradan okur.
- Kalıcı kopya Drive'da `firesmoke/raw_zips/`: FASDD tar'ları, Boreal zip'leri. Hugging Face aynası üçüncü kişiye ait ve kaldırılabilir; Drive yedeği bu riske karşı.
- Sonraki oturumlarda FASDD, varsa Drive yedeğinden alınır; yoksa Hugging Face'ten iner.

**Bu notebook ile `02_audit` aynı oturumda çalıştırılır.** Hücreleri sırayla, tek tek çalıştır; her hücrenin altındaki notu oku.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, subprocess
from pathlib import Path
from collections import Counter

ROOT = Path('/content/drive/MyDrive/firesmoke')   # kalıcı (Drive)
RAW  = Path('/content/raw')                       # geçici, hızlı (oturum diski)
RAW.mkdir(parents=True, exist_ok=True)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

print('Drive klasörleri:', sorted(p.name for p in ROOT.iterdir()))

In [ ]:
!df -h /content | tail -1
!df -h /content/drive | tail -1

İlk satır oturum diski: en az ~45 GB boş olmalı (FASDD 15 GB tar + 15 GB açılmış hali, Boreal 12 GB zip + açılmış hali ayrıca). İkinci satır Drive'ın.

---
## 1. FASDD_UAV

Kaynak: Hugging Face aynası (`annotations.tar` 231 MB + `images.tar` 15.1 GB). Resmi kaynak Science Data Bank; ayna üçüncü kişi kopyası olduğu için aşağıda sayıları resmi değerlerle karşılaştırıyoruz.

In [ ]:
!pip -q install -U huggingface_hub hf_xet
import shutil
from huggingface_hub import hf_hub_download

REPO = 'hanvithSai/gavin-fasdd-uav-raw'
FILES = ['MANIFEST.md', 'annotations.tar', 'images.tar']
DL = RAW / 'fasdd_dl'
DL.mkdir(parents=True, exist_ok=True)
BACKUP = ROOT / 'raw_zips' / 'fasdd_uav'          # Drive'daki kalıcı yedek
BACKUP.mkdir(parents=True, exist_ok=True)

for fname in FILES:
    if (BACKUP / fname).exists():
        shutil.copy(BACKUP / fname, DL / fname)
        how = 'Drive yedeği'
    else:
        hf_hub_download(repo_id=REPO, repo_type='dataset', filename=fname, local_dir=str(DL))
        how = 'Hugging Face'
    print(f'{fname:<16} {(DL / fname).stat().st_size / 1e9:6.2f} GB  <- {how}')

### Drive'a yedekle (sadece ilk çalıştırmada iş yapar)

Yedek zaten varsa hücre hiçbir şey kopyalamaz. 15 GB'ın Drive'a yazılması uzun sürebilir; bitmesini bekle, **bu hücre bitmeden sonrakini çalıştırma** — bir sonraki hücre `/content`'teki `images.tar`'ı siliyor.

In [ ]:
for fname in FILES:
    dst = BACKUP / fname
    if not dst.exists():
        shutil.copy(DL / fname, dst)
        print('yedeklendi:', dst.name)
    else:
        print('zaten var :', dst.name)
!ls -lh {BACKUP}

In [ ]:
F = RAW / 'fasdd_uav'
F.mkdir(exist_ok=True)
!tar -xf {DL}/annotations.tar -C {F}
!tar -xf {DL}/images.tar -C {F} && rm {DL}/images.tar
!find {F} -maxdepth 3 -type d | head -20

`images.tar` açıldıktan sonra `/content`'ten siliniyor (diskte 15 GB yer açar); Drive yedeği yerinde kalır. Son satır klasör yapısını gösterir: `images/` ve `annotations/YOLO_UAV/` görmelisin.

### macOS hayalet dosyalarını sil

Bu tar macOS'ta paketlenmiş: her gerçek dosyanın yanında `._` ile başlayan ikili bir metadata dosyası var (`._fire_UAV000001.txt` gibi). Etiket sanılıp okunurken `UnicodeDecodeError` verirler ve sayıları iki katına çıkarırlar. Sadece Colab'daki kopyadan silinir.

In [ ]:
def remove_mac_junk(base):
    junk = [p for p in base.rglob('._*') if p.is_file()]
    for p in junk:
        p.unlink()
    macosx = [p for p in base.rglob('__MACOSX') if p.is_dir()]
    for p in macosx:
        shutil.rmtree(p)
    print(f'{len(junk)} adet ._ dosyası, {len(macosx)} adet __MACOSX klasörü silindi ({base.name})')

import shutil
remove_mac_junk(RAW / 'fasdd_uav')

In [ ]:
F = RAW / 'fasdd_uav'

def find_dir(base, name):
    hits = [p for p in base.rglob(name) if p.is_dir()]
    assert hits, f'{name} bulunamadı: {base}'
    return min(hits, key=lambda p: len(p.parts))   # en üstteki eşleşme

IMG_DIR  = find_dir(F, 'images')
YOLO_DIR = find_dir(F, 'YOLO_UAV')
LBL_DIR  = YOLO_DIR / 'labels'

f_imgs = [p for p in IMG_DIR.rglob('*') if p.suffix.lower() in IMG_EXT]
f_lbls = list(LBL_DIR.glob('*.txt'))
print(f'görüntü    : {len(f_imgs):>6}  (beklenen 25097)')
print(f'YOLO etiket: {len(f_lbls):>6}')
expected = {'train': 12551, 'val': 8365, 'test': 4181}
for s, n_exp in expected.items():
    lines = (YOLO_DIR / f'{s}.txt').read_text().split()
    print(f'{s:<5}: {len(lines):>6}  (beklenen {n_exp})  örnek satır: {lines[0]}')

In [ ]:
img_stems = {p.stem for p in f_imgs}
lbl_stems = {p.stem for p in f_lbls}
empty = sum(1 for p in f_lbls if not p.read_text().strip())
print('etiketi olmayan görüntü :', len(img_stems - lbl_stems))
print('görüntüsü olmayan etiket:', len(lbl_stems - img_stems))
print('boş etiket (negatif)    :', empty)

### Sınıf indeksini kendin doğrula

Dosya adı öneki görüntünün içeriğini söylüyor (`fire_`, `smoke_`, `bothFireAndSmoke_`, `neitherFireNorSmoke_`). Etiketlerdeki sınıf numaralarını bu öneklerle karşılaştırıyoruz; böylece "0=fire, 1=smoke" bilgisini dokümana güvenmeden veriden kanıtlamış oluyoruz.

In [ ]:
def class_ids(prefix, k=500):
    c = Counter()
    files = [p for p in f_lbls if p.stem.startswith(prefix)][:k]
    for p in files:
        for line in p.read_text().splitlines():
            if line.strip():
                c[line.split()[0]] += 1
    return len(files), dict(c)

for pre in ['fire_', 'smoke_', 'bothFireAndSmoke_', 'neitherFireNorSmoke_']:
    n, c = class_ids(pre)
    print(f'{pre:<22} {n:>4} dosya   sınıf sayımı: {c}')

Beklenen: `fire_` → yalnızca `'0'`, `smoke_` → yalnızca `'1'`, `bothFireAndSmoke_` → ikisi, `neitherFireNorSmoke_` → boş `{}`.
**Tutmuyorsa dur ve çıktıyı bana yapıştır.**

---
## 2. D-Fire — sadece negatifler

Kaynak: D-Fire README'sinden linkli Kaggle sürümü. Pozitif görüntüleri almıyoruz (çoğu yer seviyesi kamera, drone değil); sadece etiketsiz, yanıltıcı negatifleri (bulut, sis, gün batımı) false positive'i düşürmek için kullanacağız.

In [ ]:
!pip -q install -U kagglehub
import kagglehub
DFIRE = Path(kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo'))
print(DFIRE)
!du -sh {DFIRE}
!find {DFIRE} -maxdepth 3 -type d
for y in DFIRE.rglob('*.yaml'):
    print('---', y.relative_to(DFIRE)); print(y.read_text()[:400])

Kimlik doğrulama isterse: kaggle.com → Settings → API → **Create New Token**. Colab'ın sol panelindeki anahtar simgesine (Secrets) `KAGGLE_USERNAME` ve `KAGGLE_KEY` ekle, notebook erişimini aç, hücreyi tekrar çalıştır.

Yaml çıktısında sınıf isimlerine bak: orijinal D-Fire'da `0=smoke, 1=fire`. Biz sadece negatif aldığımız için şimdilik fark etmiyor ama not et.

In [ ]:
def label_path(img):
    # .../images/.../x.jpg  ->  .../labels/.../x.txt  (son 'images' klasörü değiştirilir)
    parts = list(img.parts)
    i = len(parts) - 1 - parts[::-1].index('images')
    parts[i] = 'labels'
    return Path(*parts).with_suffix('.txt')

def is_negative(img):
    lp = label_path(img)
    return (not lp.exists()) or lp.read_text().strip() == ''

d_imgs = [p for p in DFIRE.rglob('*') if p.suffix.lower() in IMG_EXT and 'images' in p.parts
          and not p.name.startswith('._')]
d_neg = [p for p in d_imgs if is_negative(p)]
no_label_file = sum(1 for p in d_imgs if not label_path(p).exists())
print(f'D-Fire görüntü: {len(d_imgs)}   negatif: {len(d_neg)} (beklenen ~9838)   etiket dosyası hiç olmayan: {no_label_file}')

In [ ]:
import shutil
NEG = RAW / 'dfire_neg' / 'images'
NEG.mkdir(parents=True, exist_ok=True)
for p in d_neg:
    rel = '_'.join(p.relative_to(DFIRE).parts)     # split bilgisini isimde koru, çakışma olmasın
    shutil.copy2(p, NEG / f'dfire_{rel}')
print(len(list(NEG.iterdir())), 'negatif kopyalandı →', NEG)

---
## 3. Boreal Forest Fire — senin yapacağın manuel adım

Fairdata'da indirme tarayıcıdan yapılıyor. Doğrudan bir indirme linki bulursan bana gönder, bu bölümü otomatikleştiririm.

1. https://etsin.fairdata.fi/dataset/1dce1023-493a-4d63-a906-f2a44f831898 → **Data** sekmesi
2. Sadece şu iki klasörü indir:
   - `Boreal-Forest-Fire-Subset-A` (bbox etiketli görüntüler)
   - `Visualisation-And-Coordinate-Conversion-Notebook` (koordinat formatının açıklaması)
   - Subset B (4K videolar) ve C (segmentasyon) **gerekmez**.
3. İnen zip'leri Drive'da `firesmoke/raw_zips/` klasörüne yükle (kök seviyesine, `fasdd_uav/` alt klasörünün içine değil), sonra aşağıdaki hücreyi çalıştır.

In [ ]:
zips = sorted((ROOT / 'raw_zips').glob('*.zip'))
print('bulunan zip:', [z.name for z in zips])
B = RAW / 'boreal'
B.mkdir(exist_ok=True)
for z in zips:
    subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(B)], check=True)
remove_mac_junk(B)
dirs = sorted({str(p.relative_to(B)) for p in B.rglob('*') if p.is_dir() and len(p.relative_to(B).parts) <= 4})
print('\n'.join(dirs[:40]))

In [ ]:
ann_ext = {'.txt', '.json', '.csv', '.xml'}
b_imgs = [p for p in B.rglob('*') if p.suffix.lower() in IMG_EXT]
b_anns = [p for p in B.rglob('*') if p.suffix.lower() in ann_ext]
print('görüntü:', len(b_imgs), '  annotation dosyası:', len(b_anns))
print('uzantılar:', dict(Counter(p.suffix.lower() for p in b_anns)))
for p in b_anns[:3]:
    print('\n===', p.relative_to(B))
    print(p.read_text(errors='replace')[:600])
if b_imgs:
    from PIL import Image
    print('\nörnek görüntü:', b_imgs[0].relative_to(B), Image.open(b_imgs[0]).size)

**Bu hücrenin çıktısını bana yapıştır.** Boreal'in koordinat formatını buradan görüp Faz 3'teki YOLO dönüştürücüsünü ona göre yazacağım.

---
## 4. Özet → Drive

In [ ]:
summary = {
    'fasdd_uav': {'images': len(f_imgs), 'labels': len(f_lbls), 'empty_labels': empty},
    'dfire': {'images': len(d_imgs), 'negatives': len(d_neg)},
    'boreal': {'images': len(globals().get('b_imgs', [])), 'annotation_files': len(globals().get('b_anns', []))},
}
(ROOT / 'reports').mkdir(exist_ok=True)
(ROOT / 'reports' / '01_download_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

**Kontrol noktası:** Görüntü/etiket sayıları beklenenle tutuyor, sınıf indeksi veriden doğrulandı, D-Fire negatifleri `/content/raw/dfire_neg` altında. Aynı oturumda `02_audit.ipynb`'ye geç.